In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter
from pathlib import PurePosixPath
import subprocess
import sys
import json

from collections import defaultdict
from io import BytesIO

from IPython.display import display
from PIL import Image, ImageDraw

In [3]:
archive_dir = Path("/content/drive/MyDrive/WasteScope/raw_archives")

archives = {
    "detection": archive_dir / "detection_task.zip",
    "tracking": archive_dir / "tracking_task.zip",
    "segmentation": archive_dir / "segmentation_task.zip",
}

In [4]:
def video_id(filename):
    return PurePosixPath(filename).stem.split("_")[0]

image_counts = {}

for task, archive in archives.items():
    with ZipFile(archive) as zf:
        image_counts[task] = Counter(
            video_id(item.filename)
            for item in zf.infolist()
            if item.filename.lower().endswith(".png")
        )

with ZipFile(archives["segmentation"]) as zf:
    with zf.open("Segmentation Task/labels.json") as file:
        coco = json.load(file)

category_names = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

image_to_video = {
    image["id"]: video_id(image["file_name"])
    for image in coco["images"]
}

In [5]:
annotated_image_ids = set()
category_instances = defaultdict(Counter)
category_images = defaultdict(lambda: defaultdict(set))

for annotation in coco["annotations"]:
    image_id = annotation["image_id"]
    category_id = annotation["category_id"]
    
    vid = image_to_video[image_id]
    annotated_image_ids.add(image_id)
    category_instances[vid][category_id] += 1
    category_images[vid][category_id].add(image_id)
    
empty_images = Counter(
    video_id(image["file_name"])
    for image in coco["images"]
    if image["id"] not in annotated_image_ids
)

print("Category ID legend:")
for category_id, name in sorted(category_names.items()):
    print(f"  {category_id}: {name}")

Category ID legend:
  1: Overflow
  2: container_yellow
  3: container_green
  4: container_default
  5: container_blue
  6: container_ash
  7: container_oil
  8: container_battery
  9: container_biodegradable


In [15]:
print("Per-video distribution (segmentation 38 video):")
print("video   detection  tracking  segmentation  overflow+  empty  categories")

for vid in sorted(image_counts["segmentation"]):
    present_categories = ",".join(
        str(category_id)
        for category_id in sorted(category_names)
        if category_instances[vid][category_id] > 0
    )

    print(
        f"{vid:6}  "
        f"{image_counts['detection'][vid]:9}  "
        f"{image_counts['tracking'][vid]:8}  "
        f"{image_counts['segmentation'][vid]:12}  "
        f"{len(category_images[vid][1]):9}  "
        f"{empty_images[vid]:5}  "
        f"{present_categories}"
    )

Per-video distribution (segmentation 38 video):
video   detection  tracking  segmentation  overflow+  empty  categories
0002           50         0            50         50      0  1,4
0003          290         0           290        290      0  1,4
0008          140         0           140          0      1  4
0012          260         0           260        260      0  1,2,3,4,5,8
0018          340        90           250        107     13  1,2,3,5
0019          440         0           440          0      0  2,3,4,5,9
0020          600         0           600        505      0  1,2,3,4,5,8,9
0023          210         0           210        142      0  1,2,3,4,5,9
0025          200         0           200          0      0  2,3,4,5,8,9
0031          170         0           170         78     19  1,2,3,4,5,7,8
0034          120         0           120         69      0  1,2,3,4,5
0035          150       100            50          0      0  2,3,5
0038           90         0            9

In [16]:
print("Category coverage across videos:")
print("ID  category                 videos  images  instances")

for category_id, name in sorted(category_names.items()):
    videos = [
        vid for vid in image_counts["segmentation"]
        if category_instances[vid][category_id] > 0
    ]
    images = sum(
        len(category_images[vid][category_id])
        for vid in videos
    )
    instances = sum(
        category_instances[vid][category_id]
        for vid in videos
    )

    print(
        f"{category_id:2}  {name:24} "
        f"{len(videos):6}  {images:6}  {instances:9}"
    )

    if len(videos) <= 10:
        print("    Video IDs:", ", ".join(sorted(videos)))

Category coverage across videos:
ID  category                 videos  images  instances
 1  Overflow                     33    4197       5149
 2  container_yellow             32    4933       5191
 3  container_green              32    4909       4909
 4  container_default            34    5700      12349
 5  container_blue               32    5144       5448
 6  container_ash                 2     212        212
    Video IDs: 0054, 0110
 7  container_oil                 7     983        983
    Video IDs: 0031, 0038, 0047, 0048, 0052, 0107, 0110
 8  container_battery            11    1247       1301
 9  container_biodegradable       5    1212       1212
    Video IDs: 0019, 0020, 0023, 0025, 0047


In [17]:
focus_categories = [6, 7, 8, 9]

for category_id in focus_categories:
    print(f"\n{category_names[category_id]} (category {category_id})")
    print("video  images  instances  total_seg_images  tracking_images")
    
    for vid in sorted(image_counts["segmentation"]):
        instances = category_instances[vid][category_id]
        if instances == 0:
            continue
        
        print(
            f"{vid:5}  "
            f"{len(category_images[vid][category_id]):6}  "
            f"{instances:9}  "
            f"{image_counts['segmentation'][vid]:16}  "
            f"{image_counts['tracking'][vid]:15}"
        )


container_ash (category 6)
video  images  instances  total_seg_images  tracking_images
0054      188        188               220                0
0110       24         24               220                0

container_oil (category 7)
video  images  instances  total_seg_images  tracking_images
0031       28         28               170                0
0038       40         40                90                0
0047       92         92               300                0
0048      237        237               260                0
0052      256        256               370                0
0107      240        240               320               41
0110       90         90               220                0

container_battery (category 8)
video  images  instances  total_seg_images  tracking_images
0012      233        233               260                0
0020      456        456               600                0
0025       55         55               200                0
0031       4

**Detection category coverage**

In [18]:
actual_class_names = {
    0: "Overflow",
    1: "container_yellow",
    2: "container_green",
    3: "container_default",
    4: "container_blue",
    5: "container_ash",
    6: "container_oil",
    7: "container_battery",
    8: "container_biodegradable",
}

actual_instances = defaultdict(Counter)
actual_images = defaultdict(lambda: defaultdict(set))
problem_counts = Counter()
problem_examples = []
empty_label_files = 0

with ZipFile(archives["detection"]) as zf:
    labels = sorted(
        (
            item for item in zf.infolist()
            if item.filename.startswith("Detection Task/labels/")
            and item.filename.endswith(".txt")
        ),
        key=lambda item: item.header_offset,
    )

    for number, item in enumerate(labels, start=1):
        vid = video_id(item.filename)
        stem = PurePosixPath(item.filename).stem
        content = zf.read(item).decode("utf-8-sig")

        if not content.strip():
            empty_label_files += 1
            continue

        for line in content.splitlines():
            if not line.strip():
                continue

            fields = line.split()

            if len(fields) != 5:
                reason = "wrong_field_count"
            else:
                try:
                    class_id = int(fields[0])
                    _box = [float(value) for value in fields[1:]]
                    reason = (
                        None if class_id in actual_class_names
                        else "unknown_class_id"
                    )
                except ValueError:
                    reason = "non_numeric_value"

            if reason is not None:
                problem_counts[reason] += 1
                if len(problem_examples) < 5:
                    problem_examples.append((item.filename, line, reason))
                continue

            actual_instances[vid][class_id] += 1
            actual_images[vid][class_id].add(stem)

        if number % 10000 == 0:
            print(f"Read {number}/{len(labels)} labels")

print(f"\nEmpty label files: {empty_label_files}")
print("Actual problem counts:", dict(problem_counts))
print("Problem examples:", problem_examples)

Read 30000/36478 labels

Empty label files: 19394
Actual problem counts: {}
Problem examples: []


In [19]:
print("Corrected detection category coverage:")
print("ID  category                  videos  images  instances")

for class_id, name in actual_class_names.items():
    videos = sorted(
        vid for vid in image_counts["detection"]
        if actual_instances[vid][class_id] > 0
    )
    images = sum(len(actual_images[vid][class_id]) for vid in videos)
    instances = sum(actual_instances[vid][class_id] for vid in videos)

    print(
        f"{class_id:2}  {name:25} "
        f"{len(videos):6}  {images:6}  {instances:9}"
    )

    if class_id in (5, 6, 7, 8):
        print(
            "    Per-video images:",
            ", ".join(
                f"{vid}:{len(actual_images[vid][class_id])}"
                for vid in videos
            )
        )

Corrected detection category coverage:
ID  category                  videos  images  instances
 0  Overflow                      33    4197       5149
 1  container_yellow              75    9768      10208
 2  container_green               78    9918       9918
 3  container_default            114   14346      28346
 4  container_blue                77   10278      10930
 5  container_ash                  5     408        408
    Per-video images: 0054:188, 0077:84, 0078:85, 0110:24, 0111:27
 6  container_oil                 15    1873       1873
    Per-video images: 0013:155, 0029:151, 0031:28, 0032:134, 0038:40, 0039:98, 0047:92, 0048:237, 0052:256, 0068:44, 0107:240, 0110:90, 0113:17, 0118:154, 0122:137
 7  container_battery             28    2535       2595
    Per-video images: 0012:233, 0013:135, 0015:135, 0017:161, 0020:456, 0025:55, 0031:47, 0032:35, 0039:41, 0040:96, 0044:24, 0047:86, 0050:62, 0056:48, 0065:91, 0068:17, 0074:44, 0084:220, 0096:10, 0100:97, 0103:109, 0107:60,